# Manual QIR Circuit Submission to Azure Quantum

This notebook demonstrates how to submit quantum circuits to Azure Quantum via an explicit **OpenQASM 3 → QIR** compilation path, covering both **Qiskit** and **Cirq** circuits.

This approach gives full visibility and control over intermediate compilation artifacts. It is useful when you need to inspect or modify OpenQASM or QIR before submission, target backends that require raw QIR, or share a single submission pipeline across frameworks.

For the recommended getting-started experience for each framework, see:
- `qiskit_submission_to_azure.ipynb`
- `cirq_submission_to_azure.ipynb`

The same compilation pipeline works for both Qiskit and Cirq circuits:

1. Build a circuit in Qiskit or Cirq.
2. Export it to **OpenQASM 3** text — this intermediate representation can be inspected or modified.
3. Compile the OpenQASM 3 source to **QIR** with `qdk.openqasm.compile` (choose a `TargetProfile`).
4. Connect to an Azure Quantum workspace with `qdk.azure.Workspace`.
5. Select a target (e.g. a simulator such as `rigetti.sim.qvm`).
6. Submit the QIR payload (`target.submit(qir, job_name, shots)`) and retrieve results (`job.get_results()`).

## Prerequisites

Install the `qdk` package with `azure` and whichever framework extras you need. Install both if you plan to run both sections of this notebook:

In [ ]:
%pip install "qdk[azure,cirq,qiskit]"

After installing, restart the notebook kernel if it was already running. Verify installation:

In [ ]:
import cirq, qiskit, qdk, qdk.azure  # should import without errors

## Configure Azure Quantum workspace connection

Replace the placeholder values with your own workspace details. These are shared across both the Qiskit and Cirq examples below.

In [ ]:
subscription_id = 'xxxxxxxx-xxxx-xxxx-xxxx-xxxxxxxxxxxx'
resource_group = 'myresourcegroup'
workspace_name = 'myworkspace'
location = 'westus'

# Replace with any QIR-capable target name from your workspace
target_name = "rigetti.sim.qvm"

---
## Part A: Submitting a Qiskit circuit

### Build a Qiskit circuit

We use a Bell-state circuit — two entangled qubits — to demonstrate the Qiskit path.


In [ ]:
from qiskit import QuantumCircuit

qiskit_circuit = QuantumCircuit(2, 2)
qiskit_circuit.h(0)
qiskit_circuit.cx(0, 1)
qiskit_circuit.measure([0, 1], [0, 1])

qiskit_circuit.draw(output="text")


### Export to OpenQASM 3

`qiskit.qasm3.dumps` converts a Qiskit circuit to an OpenQASM 3 string. This intermediate text can be inspected or modified before compilation.

In [ ]:
from qiskit import qasm3

qasm3_str_qiskit = qasm3.dumps(qiskit_circuit)
print(qasm3_str_qiskit)

### Compile to QIR and submit

Compile the OpenQASM 3 source to QIR, then connect to the workspace and submit the payload.

In [ ]:
from qdk.openqasm import compile
from qdk import TargetProfile
from qdk.azure import Workspace

qir_qiskit = compile(qasm3_str_qiskit, target_profile=TargetProfile.Base)

workspace = Workspace(
    subscription_id=subscription_id,
    resource_group=resource_group,
    name=workspace_name,
    location=location,
)

target = workspace.get_targets(target_name)
job = target.submit(qir_qiskit, "qiskit-manual-job", shots=100)
print("Job submitted. Waiting for results...")

results = job.get_results()
print(results)

---
## Part B: Submitting a Cirq circuit

### Build a Cirq circuit

We create a simple Bell-state circuit with two named measurement keys.

In [ ]:
import cirq

q0, q1 = cirq.LineQubit.range(2)
cirq_circuit = cirq.Circuit(
    cirq.H(q0),
    cirq.CNOT(q0, q1),
    cirq.measure(q0, key="q0"),
    cirq.measure(q1, key="q1"),
)
print(cirq_circuit)

### Export to OpenQASM 3

`circuit.to_qasm(version="3.0")` converts a Cirq circuit to an OpenQASM 3 string. This intermediate text can be inspected or modified before compilation.

In [ ]:
qasm3_str_cirq = cirq_circuit.to_qasm(version="3.0")
print(qasm3_str_cirq)

### Compile to QIR and submit

The compile and submission steps are identical to Part A — `compile` and `workspace` are already available from the cells above.

In [ ]:
qir_cirq = compile(qasm3_str_cirq, target_profile=TargetProfile.Base)

target = workspace.get_targets(target_name)
job = target.submit(qir_cirq, "cirq-manual-job", shots=100)
print("Job submitted. Waiting for results...")

results = job.get_results()
print(results)

## Notes

- **TargetProfile**: Use `TargetProfile.Base` for most simulators and hardware targets. Use `TargetProfile.Adaptive_RI` for targets that support adaptive circuits with classical branching.
- **Shared compilation step**: The `compile → submit` pipeline is identical for both frameworks — the only difference is how the OpenQASM 3 string is produced.
- **QIR inspection**: The `qir` object is LLVM bitcode. You can write it to a `.bc` file for offline inspection with LLVM tools.
